# Quantum Teleportation

Commented revision of `qc-lab4.ipynb`, prepared 2026-11-06. Run cells from top to bottom. Qiskit displays classical bit strings with the highest-index bit on the left. Original import copies have outputs cleared and embedded credentials removed.

Alice uses q[1] and q[2]; Bob receives the state on q[3]. q[0] is unused in the original layout. Bob applies X for c[2] and Z for c[1]. The final counts should give approximately 75% zero and 25% one on Bob’s qubit.

In [ ]:
from qiskit import QuantumRegister,ClassicalRegister,QuantumCircuit

In [ ]:
q = QuantumRegister(4,'q')
c = ClassicalRegister(4,'c')

In [ ]:
qc = QuantumCircuit(q,c)

In [ ]:
# unknown state
# qc.x(q[1]) this makes ket 1

import numpy as np

psi = [np.sqrt(3)/2, 1/2]

qc.initialize(psi, q[1])  # our own si value 

qc.barrier()

#psi = [alpha, beta]


In [ ]:
qc.h(q[2])
qc.cx(q[2],q[3])
qc.barrier()

In [ ]:
qc.draw()

In [ ]:
# ---------------------------------------
# STEP 3: Alice's Bell measurement
# ---------------------------------------

qc.cx(q[1], q[2])
qc.h(q[1])

qc.barrier()
qc.draw()

In [ ]:
# Measure Alice's two qubits
qc.measure(q[1], c[1])
qc.measure(q[2], c[2])

qc.draw()

In [ ]:
# ---------------------------------------
# STEP 4: Bob applies corrections
# ---------------------------------------

# If measurement of q[2] = 1 → apply X
with qc.if_test((c[2], 1)):
    qc.x(q[3])

# If measurement of q[1] = 1 → apply Z
with qc.if_test((c[1], 1)):
    qc.z(q[3])
    
qc.draw()


In [ ]:
qc.draw('mpl')

In [ ]:
from qiskit_aer import AerSimulator

# Create simulator
simulator = AerSimulator()

# Run circuit 1000 times
job = simulator.run(qc, shots=1000)

# Get result
result = job.result()

# Get counts
counts = result.get_counts(qc)

print(counts)

In [ ]:
qc.barrier()

qc.measure(q[3], c[3])

qc.draw('mpl')

In [ ]:
from qiskit_aer import AerSimulator
from qiskit import transpile

simulator = AerSimulator()

compiled_qc = transpile(qc, simulator)

job = simulator.run(compiled_qc, shots=1000)

result = job.result()

counts = result.get_counts()

print(counts)

In [ ]:
counts

In [ ]:
total_0 = 0
total_1 = 0
for i in counts:
    if int(i[0]) == 1:
        total_1 += counts[i]
    if int(i[0]) == 0:
        total_0 += counts[i]
print(total_0)
print(total_1)
        
        

In [ ]:
# Verify the full quantum state before Bob's final measurement.
# Invert its preparation and check that Bob returns to |0>.
verification = qc.copy()
# Remove only Bob's terminal measurement, preserving Alice's measurements.
verification.data = [inst for inst in verification.data if not (inst.operation.name == "measure" and verification.find_bit(inst.qubits[0]).index == 3)]
verification.ry(-2 * np.arcsin(psi[1]), 3)
verification.save_density_matrix([3])
verification_backend = AerSimulator(method="density_matrix")
verification_result = verification_backend.run(transpile(verification, verification_backend), shots=128, seed_simulator=42).result()
fidelity = float(np.real(verification_result.data(0)["density_matrix"][0, 0]))
assert np.isclose(fidelity, 1.0), fidelity
print("Teleportation fidelity:", fidelity)